# Local Machine Snowflake Connection

In [1]:
import pandas as pd
import numpy as np
#import pyodbc
import glob
import os
import re
import snowflake.connector
import urllib.parse
from sqlalchemy import create_engine, text
from getpass import getpass
from openpyxl import load_workbook

In [2]:
user_nm = 'neha.yadav1'

In [3]:
sfAccount: str =  'merkle-amica' 
sfauthenticator: str = 'externalbrowser'

ctx = snowflake.connector.connect(
    user= f'{user_nm}@merkleinc.com', #enter your user name
    account=sfAccount,
    authenticator=sfauthenticator,
    role = 'AMICA_SNOWFLAKE_PROD_NON_PII_CAMPAIGN' #other options are: 'AMICA_SNOWFLAKE_PROD_NON_PII_CAMPAIGN', 'AMICA_SNOWFLAKE_PROD_PII_ANALYTICS', 'AMICA_SNOWFLAKE_UAT_PII_CAMPAIGN'
    )

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://myapps.dentsu.com/app/snowflake/exkdty4rr8nj82z0H0i7/sso/saml?SAMLRequest=lVJdb%2BIwEPwrke85sQkfpRZQpUVVOdGWA1rp7s1NFjBx7JzXIXC%2F%2Fhw%2Bqt5DK92bvZ7xzO7s4GZfqGAHFqXRQ9KKGAlApyaTej0kL8v7sE8CdEJnQhkNQ3IAJDejAYpClTyp3EbP4XcF6AL%2FkUbePAxJZTU3AiVyLQpA7lK%2BSB6nPI4YF4hgnZcjZ0qG0mttnCs5pXVdR3U7MnZNY8YYZdfUoxrIN%2FJBovxao7TGmdSoC2Xve%2FpEokVZp5HwCK8wOxNvpT6N4CuVtxMI%2BcNyOQtnz4slCZJLd3dGY1WAXYDdyRRe5tOTAfQOfDlXEIpCpiJCbeqVEjmkpigr5z%2BM%2FImuIKPKrKUf02Q8JGUus%2F3ucdu%2F7rTjx1fWxu42%2F%2BG684POkzJ%2Fmm%2BSW5F%2Bn%2FaufnbfNpOUBK%2BXUOMm1AliBRPdROl8icW9kPXCuLNsxbzb53EctTvsFwnGPkqphTsy3%2F0eRFlilIF2WB3d%2BTt9N05hn2fu0LG2r7f9%2BA97YPKKIhraREVO28KPDuzoP2YwoB%2BJ55178jFMxjOjZHoI7o0thPs8pVbUOlZkFq6OUA6FkCrJMguIPi2lTH1nQTi%2F2s5WQOjopPrvco%2F%2BAg%

Enter the URL the SSO URL redirected you to:  http://localhost:51471/?token=7Vlbd6LKEn73V2Q5j66Ei4DommQfrooKCoIIb1xaRK5y119_wIyZJDM7M3tmr_Nw1uTFdHVVdfVXRfGVfv6ricK7CmS5n8SPfeQB7t-B2ElcP_Ye-5rK35P9v54-51YUoulEAXmaxDm4Y0Fe-LFVXI0ORZHmEwgSQRaE4J4SBYZ6yOOk3odWAJwkSstW2Xto_4P2wIXCxPPj_p3APvZ99x4fk9gYhwlyiJMIMhrCY6T7Gw8xlMBarfh2qJo89tPAd5tKPJJjbIiKW3iY48dALnDlHAdUGkjKgaItZ74kRgZuHwSntc_zEghxXlhx8dhHYZS4h4l7FFMRdIKTExR_QAjE7N9tbxCgHQQtKHE-eb71Y7_M4kli5X4-ia0I5JPCmWwocTlpVSdplhSJk4Q3myZ_BqTFo67rh3r4kGQehMIwAu3E5cY5gMjqfwF0cg0uu-OTLLKKj8_pJC1a-6vqBMSFX5zfxPmxuZXnIOvS1X96FV0SFNY1LaAJ3OKMZRkZH0n0As9gf_QZeh3k02c3n2x8r016mYEvB7t_d1kYgsdQq-Pmvvep_2ILXCHeJ9clY8VJ7DtW6F-uZSSC4pC4d1ToJZlfHKK_RRGBO8f3oHHuHQSLP_Wht6H9tCMYu0V4HyUZ-JTl1n1-sFCc-OJSAXuQtQ8DuNMU4bH_6YfFerVSMyvOuzTlb5c_jOgNZiCuQJikwL3Pbxf7EtXPO_wOVk-fgTMRYicsc78CUlcmqeWA_G6dgb3fLP28LcMmvxUWcP5RGqDX4b1bPqPB-l7bOf5hitoUfHqTmGcvWysswRNPqqSpaZuwZsiEdpCUUKG9STgrDqEC4Yhl6nYN79Lq3LaPa0Svja-ClzQ_L9_V6UtdPVt4C2KwGzhYCVlYtLLt4nDm3RVwpE2qFUIQO2ssl087rymB

In [4]:
cur = ctx.cursor()

cur.execute('USE WAREHOUSE AMICA_WH_XSMALL') 
cur.execute("USE DATABASE FIVETRAN_AMICA_DATABASE")
cur.execute("USE SCHEMA AGGREGATE")

In [7]:

# Define your date range variables
start_date = '01-Jan-2025'
end_date = '20-jun-2026'



# Define date range variables
start_date1 = '2025-01-01'
end_date1 = '2026-06-20'



In [8]:
query = f'''


SELECT DATE, year, month, state, line, engine, keyword_type, campaign_name, campaign_type,
       SUM(impressions) AS impressions, SUM(spend) AS spend, SUM(clicks) AS clicks, SUM(conversions) AS conversions
FROM (
    SELECT
        stg_source,
        date,
        YEAR(date) AS year,
        MONTH(date) AS month,
        state_code AS state,
        campaign_name,
        campaign_type,
        CASE 
            WHEN UPPER(campaign_type) LIKE '%DEMAND_GEN' OR UPPER(campaign_type) LIKE '%VIDEO' THEN 'YouTube'
            WHEN UPPER(campaign_type) LIKE '%PERFORMANCE MAX' OR UPPER(campaign_type) LIKE '%SEARCH' THEN 'Google Search'
            ELSE campaign_type END AS engine,
        CASE
            WHEN UPPER(campaign_name) LIKE '%AUTO%' THEN 'Auto'
            WHEN UPPER(campaign_name) LIKE '%HOME%' THEN 'Home'
            WHEN UPPER(campaign_name) LIKE '%CONDO%' THEN 'Condo'
            WHEN UPPER(campaign_name) LIKE '%RENTERS%' THEN 'Renters'
            WHEN UPPER(campaign_name) LIKE '%ENTERPRISE%' OR UPPER(campaign_name) LIKE '%BRAND%' THEN 'Enterprise'
            WHEN UPPER(campaign_name) LIKE '%LIFE%' THEN 'Life'
            ELSE 'Enterprise'
        END AS line,
        CASE
            WHEN campaign_type = 'Search' THEN 
                CASE
                    WHEN UPPER(campaign_name) LIKE '%PERFORMANCE MAX%' THEN 'PMAX'
                    WHEN UPPER(campaign_name) LIKE '%BRAND%' THEN 'Brand'
                    WHEN UPPER(campaign_name) LIKE '%GENERIC%' THEN 'Generic'
                    ELSE 'None'
                END
            WHEN campaign_type = 'Performance Max' AND UPPER(campaign_name) LIKE '%PERFORMANCE MAX%' THEN 'PMAX'
            ELSE 'None'
        END AS keyword_type,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        SUM(cost) AS spend,
        SUM(conversions) AS conversions
    FROM FIVETRAN_AMICA_DATABASE.AGGREGATE.AGG_GOOGLE
    WHERE date BETWEEN '{start_date}' AND '{end_date}'
      AND engine <> 'YouTube'
      AND line <> 'Life'
      AND UPPER(campaign_name) NOT LIKE '%LIFE%'
      AND UPPER(campaign_name) NOT IN (
          'QS_AUTO_INSURANCE','MA_AUTO_INSURANCE','MA_HOME_INSURANCE','CA_AUTO_INSURANCE','USNEWS_HOME_INSURANCE',
          'CNBC','MA_HOME_INSURANCE','MA_HOME_INSURANCE','CA_HOME_INSURANCE','MONEYGROUP_TIER1PUBS',
          'BEST_MONEY_AUTO_INSURANCE','MONEY.COM','2501R','QS_AUTO_INSURANCE'
      )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
)
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9;


'''



google_data = pd.read_sql(query, ctx)
print(google_data.shape)
google_data.head()

/tmp/ipykernel_3198170/3589701959.py:62: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  google_data = pd.read_sql(query, ctx)


(99705, 13)


,DATE,YEAR,MONTH,STATE,LINE,ENGINE,KEYWORD_TYPE,CAMPAIGN_NAME,CAMPAIGN_TYPE,IMPRESSIONS,SPEND,CLICKS,CONVERSIONS
0,2025-02-22,2025,2,VT,Condo,Google Search,Generic,Condo Insurance - Generic,SEARCH,0,0.00,0,0.0
1,2025-10-05,2025,10,ME,Auto,Google Search,Generic,Auto Generic - ME,SEARCH,83,879.01,4,2.5
2,2025-12-16,2025,12,WA,Enterprise,Google Search,Brand,Amica Brand - Enterprise - Strategic,SEARCH,148,801.20,61,15.0
3,2025-10-18,2025,10,ME,Auto,Google Search,Generic,Auto Generic - ME,SEARCH,105,1529.67,7,3.0
4,2025-11-14,2025,11,WV,Enterprise,Google Search,Brand,Amica Brand - Enterprise - Non-Strategic,SEARCH,9,48.81,3,0.0


In [9]:
google_data['Engine'] = 'Google'

In [10]:
query1 = f'''


SELECT yr_mth, state, line, engine, campaign_name, keyword_type,
       SUM(impressions) AS impressions, SUM(clicks) AS clicks, SUM(spend) AS spend, SUM(conversions) AS conversions
FROM (
    SELECT 
        stg_source,
        TO_CHAR(date,'YYYY-MM') AS yr_mth,
        state_code AS state,
        'Bing' AS engine,
        NULL AS campaign_type,
        campaign_name,
        CASE
            WHEN UPPER(campaign_name) LIKE '%AUTO%' THEN 'Auto'
            WHEN UPPER(campaign_name) LIKE '%HOME%' THEN 'Home'
            WHEN UPPER(campaign_name) LIKE '%CONDO%' THEN 'Condo'
            WHEN UPPER(campaign_name) LIKE '%RENTERS%' THEN 'Renters'
            WHEN UPPER(campaign_name) LIKE '%ENTERPRISE%' THEN 'Enterprise'
            WHEN UPPER(campaign_name) LIKE '%LIFE%' THEN 'Life'
        END AS line,
        CASE 
            WHEN UPPER(campaign_name) LIKE '%BRAND%' THEN 'Brand'
            WHEN UPPER(campaign_name) LIKE '%GENERIC%' THEN 'Generic'
        END AS keyword_type,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        SUM(spend) AS spend,
        SUM(conversions) AS conversions
    FROM FIVETRAN_AMICA_DATABASE.AGGREGATE.AGG_BING
    WHERE date BETWEEN '{start_date}' AND '{end_date}'
      AND line <> 'Life'
      AND UPPER(campaign_name) NOT LIKE '%LIFE%'
      AND UPPER(campaign_name) NOT IN (
          'QS_AUTO_INSURANCE','MA_AUTO_INSURANCE','MA_HOME_INSURANCE','CA_AUTO_INSURANCE','USNEWS_HOME_INSURANCE',
          'CNBC','MA_HOME_INSURANCE','MA_HOME_INSURANCE','CA_HOME_INSURANCE','MONEYGROUP_TIER1PUBS',
          'BEST_MONEY_AUTO_INSURANCE','MONEY.COM','2502C','2501R','QS_AUTO_INSURANCE'
      )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
)
GROUP BY 1, 2, 3, 4, 5, 6;

'''

bing_data = pd.read_sql(query1, ctx)
print(bing_data.shape)
bing_data.head()

/tmp/ipykernel_3198170/1293090693.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bing_data = pd.read_sql(query1, ctx)


(3555, 10)


,YR_MTH,STATE,LINE,ENGINE,CAMPAIGN_NAME,KEYWORD_TYPE,IMPRESSIONS,CLICKS,SPEND,CONVERSIONS
0,2025-02,RI,Auto,Bing,Amica Auto - Brand - Bing,Brand,168,36,1003.57,8
1,2026-01,IA,Auto,Bing,Amica Auto - Brand - Bing,Brand,30,2,41.59,0
2,2025-01,PA,Home,Bing,Home Generic - PA - Bing,Generic,21947,408,13022.05,99
3,2025-08,SC,Home,Bing,Amica Brand - Home Insurance - Bing,Brand,22,0,0.00,0
4,2026-03,IA,Enterprise,Bing,Amica Brand - Enterprise - Non Strategic - Bing,Brand,136,28,180.78,6


In [11]:
bing_data['Engine'] = 'Bing'

In [12]:
### internal_data

In [13]:
query2 = f'''

select a.lead_storage_nbr, a.line_of_business,a.new_segment_reporting_cd as quality_seg,
a.lead_dt, a.lead_contact_method,
a.lead_medium,b.medium_channel,b.medium,b.source,b.campaign,
sum(a.lead)lead,sum(quote)quote,sum(a.issued_policy)issue,
sum(case when lead_dt=current_proc_dt then issued_policy else 0 end ) Day0,
sum(case when current_proc_dt - lead_dt <=6 then issued_policy else 0 end ) Day7,
state_cd,
case when state_cd in('CT','ME','NY','PA','RI','VT','NH','MA') then 'Target'
     when state_cd in('AZ','DC','DE','MD','NC','NM','OR','VA','WA') then 'Tier2'
else 'Non-Target' 
end state_grp,
to_char(lead_dt,'YYYY-MM') lead_monyr
 
from  PROD_DB_PII_ANALYTICS.SANDBOX.leads_policies_auto_home_pup  a,
PROD_DB_PII_ANALYTICS.SANDBOX.last_touch_attribution_vq b
where a.lead_Dt between'{start_date}' AND '{end_date}'
and b.medium in('cpc','Paid Search')
and a.lead_storage_nbr=b.orig_storage_no
group by 1,2,3,4,5,6,7,8,9,10,16,17,18;

'''

internal_data = pd.read_sql(query2, ctx)
print(internal_data.shape)
internal_data.head()

/tmp/ipykernel_3198170/1854016874.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  internal_data = pd.read_sql(query2, ctx)


(436217, 18)


,LEAD_STORAGE_NBR,LINE_OF_BUSINESS,QUALITY_SEG,LEAD_DT,LEAD_CONTACT_METHOD,LEAD_MEDIUM,MEDIUM_CHANNEL,MEDIUM,SOURCE,CAMPAIGN,LEAD,QUOTE,ISSUE,DAY0,DAY7,STATE_CD,STATE_GRP,LEAD_MONYR
0,W011391783,AUTOMOBILE,5,2025-02-21,WEB,Paid Search,Paid Search,cpc,google,auto generic ct performance max,1,1,0,0,0,CT,Target,2025-02
1,W012273074,AUTOMOBILE,1,2025-09-30,WEB,Paid Search,Paid Search,cpc,google,auto generic - ny,1,1,0,0,0,NY,Target,2025-09
2,W011950241,AUTOMOBILE,1,2025-07-14,WEB,Paid Search,Paid Search,cpc,google,home generic - ct,1,1,0,0,0,NY,Target,2025-07
3,W013279044,AUTOMOBILE,5,2026-06-14,WEB,Paid Search,Paid Search,cpc,google,auto generic - az,1,1,0,0,0,AZ,Tier2,2026-06
4,W012467296,AUTOMOBILE,1,2025-11-18,WEB,Paid Search,Paid Search,cpc,google,enterprise - strategic,1,1,1,0,0,WA,Tier2,2025-11


In [14]:
internal_data.columns = internal_data.columns.str.lower()

In [15]:
exclude_list = [
    'QS_AUTO_INSURANCE', 'MA_AUTO_INSURANCE', 'MA_HOME_INSURANCE', 'CA_AUTO_INSURANCE',
    'USNEWS_HOME_INSURANCE', 'CNBC', 'CA_HOME_INSURANCE', 'MONEYGROUP_TIER1PUBS',
    'BEST_MONEY_AUTO_INSURANCE', 'MONEY.COM', '2501R'
]

internal_data = internal_data[
    ~internal_data['campaign'].str.upper().isin(exclude_list)
]

In [16]:
pd.set_option('display.max_columns', None)

### Web Leads

In [17]:
internal_data = internal_data[internal_data['lead_contact_method'] == 'WEB']

In [18]:

internal_data['Engine'] = internal_data.apply(
    lambda row: (
        'bing'
        # safe check: convert campaign to string, then lowercase
        if 'bing' in str(row['campaign']).lower()
        else row['source'] if row['source'] in ['google', 'bing']
        else 'google' if row['source'] == 'google.com'
        else 'bing' if row['source'] == 'bing.com'
        else 'google'
    ),
    axis=1
)


In [19]:
start_date1 = pd.to_datetime(start_date1)
end_date1 = pd.to_datetime(end_date1)
internal_data['lead_dt'] = pd.to_datetime(internal_data['lead_dt'])
internal_data = internal_data[(internal_data['lead_dt'] >= start_date1) &(internal_data['lead_dt'] <= end_date1)]
internal_data = internal_data[internal_data['medium'].isin(['Paid Search', 'cpc'])]

In [20]:
google_data['DATE'] = pd.to_datetime(google_data['DATE'])
google_data['lead_monyr'] = google_data['DATE'].dt.strftime('%Y-%m')

In [21]:
google_data.rename(columns={'STATE': 'state'}, inplace=True)
bing_data.rename(columns={'YR_MTH': 'lead_monyr', 'STATE': 'state'}, inplace=True)
internal_data.rename(columns={'state_cd': 'state'}, inplace=True)


In [22]:
internal_data['lead_dt'] = pd.to_datetime(internal_data['lead_dt'])
internal_data['lead_monyr'] = internal_data['lead_dt'].dt.strftime('%Y-%m')

In [23]:
lookup_df = pd.read_excel('/data/Users/Shelly/Search_Report/Search_camp_lookup.xlsx')

In [24]:
internal_data['campaign'] = internal_data['campaign'].astype(str).str.upper()

In [25]:
def classify_keyword(campaign):
    campaign_upper = str(campaign).upper()
    if 'BRAND' in campaign_upper or 'ENTERPRISE' in campaign_upper:
        return 'Brand'
    elif 'MAX' in campaign_upper:
        return 'PMAX'
    else:
        return 'Generic'

internal_data['keyword_type'] = internal_data['campaign'].apply(classify_keyword)

In [26]:
google_data['keyword_type'] = google_data['CAMPAIGN_NAME'].apply(classify_keyword)

In [27]:
bing_data['keyword_type'] = bing_data['CAMPAIGN_NAME'].apply(classify_keyword)

In [28]:
internal_data['Auto_Phone_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE' else None,axis=1)
internal_data['Auto_Web_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB' else None,axis=1)
internal_data['Seg_1_2_Auto_Web_leads'] = internal_data.apply(lambda row: row['Auto_Web_leads'] if row['quality_seg'] in [1, 2] else 0,axis=1)
internal_data['Seg_0_Auto_Web_leads'] = internal_data.apply(lambda row: 1 if (row['line_of_business'] == 'AUTOMOBILE' and row['quality_seg'] == 0 and row['lead_contact_method'] == 'WEB') else 0,axis=1)
internal_data['Seg_1_2_Auto_Phone_leads'] = internal_data.apply(lambda row: row['Auto_Phone_leads'] if row['quality_seg'] in [1, 2] else 0,axis=1)
internal_data['Seg_0_Auto_Phone_leads'] = internal_data.apply(lambda row: 1 if (row['line_of_business'] == 'AUTOMOBILE' and row['quality_seg'] == 0 and row['lead_contact_method'] == 'PHONE') else 0,axis=1)
internal_data['Auto_Issues'] = internal_data.apply(lambda row: row['issue'] if row['line_of_business'] == 'AUTOMOBILE' else None,axis=1)
internal_data['Auto_Web_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB') else None, axis=1)
internal_data['Auto_Phone_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE') else None, axis=1)
internal_data['Auto_Day7'] = internal_data.apply(lambda row: row['day7'] if row['line_of_business'] == 'AUTOMOBILE' else None,axis=1)
internal_data['Auto_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Auto_Phone_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE') else None,axis=1)

In [29]:
internal_data['Home_Phone_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE' else None,axis=1)
internal_data['Home_Web_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB' else None,axis=1)
internal_data['Home_Issues'] = internal_data.apply(lambda row: row['issue'] if row['line_of_business'] == 'HOMEOWNERS' else None,axis=1)
internal_data['Home_Web_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Home_Phone_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE') else None,axis=1)
internal_data['Home_Day7'] = internal_data.apply(lambda row: row['day7'] if row['line_of_business'] == 'HOMEOWNERS' else None,axis=1)
internal_data['Home_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Home_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE') else None,axis=1)

In [30]:
# Create a dictionary from the lookup table
lookup_dict = dict(zip(lookup_df['int_names'], lookup_df['eng_names']))

# Replace campaign names using the dictionary
internal_data['campaign'] = internal_data['campaign'].apply(lambda x: lookup_dict.get(x, x))

In [31]:
google_data['Engine'] = google_data['Engine'].astype(str).str.upper()
bing_data['Engine'] = bing_data['Engine'].astype(str).str.upper()

In [32]:
# Group Google Ads data
google_grouped = google_data.groupby(['lead_monyr', 'state', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
    'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

# Group Google Ads data
bing_grouped = bing_data.groupby(['lead_monyr', 'state', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
    'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

# Group Internal Company data
internal_grouped = internal_data.groupby(['lead_monyr', 'state', 'campaign','keyword_type','Engine'], as_index=False).agg({
    'lead': 'sum',
    'quote': 'sum',
    'issue': 'sum',
    'day0': 'sum',
    'day7': 'sum',
    'Auto_Phone_leads' :'sum',
    'Auto_Web_leads':'sum',
    'Seg_1_2_Auto_Web_leads':'sum',
    'Seg_0_Auto_Web_leads':'sum',
    'Seg_1_2_Auto_Phone_leads':'sum',
    'Seg_0_Auto_Phone_leads':'sum',
    'Auto_Issues':'sum',
    'Home_Phone_leads':'sum',
    'Home_Web_leads':'sum',
    'Home_Issues':'sum',
    'Auto_Day7':'sum',
    'Home_Day7':'sum',
    'Auto_Web_Issues' :'sum',
    'Auto_Phone_Issues' :'sum',
    'Auto_Web_Day7' :'sum',
    'Auto_Web_Day7' :'sum',
    'Home_Web_Issues' :'sum',
    'Home_Phone_Issues' :'sum',
    'Home_Web_Day7' :'sum',
    'Home_Web_Day7' :'sum'
    
})


In [33]:
engine = pd.concat([google_grouped, bing_grouped], axis=0, ignore_index=True)

In [34]:
lookup_df1 = pd.read_excel('/data/Users/Shelly/Search_Report/Search_camp_lookup1.xlsx')

In [35]:
# Convert campaign names to uppercase
engine['CAMPAIGN_NAME'] = engine['CAMPAIGN_NAME'].str.upper()
engine['Engine'] = engine['Engine'].str.upper()
internal_grouped['campaign'] = internal_grouped['campaign'].str.upper()
internal_grouped['Engine'] = internal_grouped['Engine'].str.upper()

In [36]:
# Create a dictionary from the lookup table
lookup_dict1 = dict(zip(lookup_df1['eng_names'], lookup_df1['int_names']))

# Replace campaign names using the dictionary
engine['CAMPAIGN_NAME'] = engine['CAMPAIGN_NAME'].apply(lambda x: lookup_dict1.get(x, x))

In [37]:
engine = engine.groupby(['lead_monyr', 'state', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
     'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})


In [38]:
exact_merged = pd.merge(engine, internal_grouped, left_on=['lead_monyr', 'state', 'CAMPAIGN_NAME', 'keyword_type', 'Engine'],right_on=['lead_monyr', 'state', 'campaign', 'keyword_type', 'Engine'], how='inner')


In [39]:

exact_merged_test = pd.merge(
    engine,
    internal_grouped,
    left_on=['lead_monyr', 'state', 'CAMPAIGN_NAME', 'keyword_type', 'Engine'],
    right_on=['lead_monyr', 'state', 'campaign', 'keyword_type', 'Engine'],
    how='outer',  # use 'outer' to capture all unmatched rows
    indicator=True
)


In [40]:
engine_unmatched = exact_merged_test[exact_merged_test['_merge'] == 'left_only']

In [41]:
internal_unmatched = exact_merged_test[exact_merged_test['_merge'] == 'right_only']

In [42]:
exact_merged = exact_merged.drop(columns=['campaign'])
engine_unmatched = engine_unmatched.drop(columns=['_merge'])
internal_unmatched = internal_unmatched.drop(columns=['_merge'])
internal_unmatched = internal_unmatched.drop(columns=['CAMPAIGN_NAME'])

In [43]:
internal_unmatched.rename(columns={
    'campaign': 'CAMPAIGN_NAME'
}, inplace=True)

In [44]:
internal_unmatched = internal_unmatched[['lead_monyr', 'state', 'CAMPAIGN_NAME', 'keyword_type', 'Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS', 'lead', 'quote', 'issue', 'day0',
       'day7', 'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
       'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
       'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
       'Home_Web_leads', 'Home_Issues', 'Auto_Day7', 'Home_Day7',
       'Auto_Web_Issues', 'Auto_Phone_Issues', 'Auto_Web_Day7',
       'Home_Web_Issues', 'Home_Phone_Issues', 'Home_Web_Day7']]


In [45]:
engine_unmatched = engine_unmatched.drop(columns=['campaign'])

In [46]:
Search_Web = pd.concat([exact_merged, engine_unmatched, internal_unmatched], ignore_index=True)

In [47]:
Search_Web = Search_Web.fillna(0)

/tmp/ipykernel_3198170/84554327.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Search_Web = Search_Web.fillna(0)


In [48]:
#Search_Web.to_csv('/data/Users/Shelly/Search Report/Search_Web.csv')

### Phone Leads

In [49]:
query2 = f'''

select a.lead_storage_nbr, a.line_of_business,a.new_segment_reporting_cd as quality_seg,
a.lead_dt, a.lead_contact_method,
a.lead_medium,b.medium_channel,b.medium,b.source,b.campaign,
sum(a.lead)lead,sum(quote)quote,sum(a.issued_policy)issue,
sum(case when lead_dt=current_proc_dt then issued_policy else 0 end ) Day0,
sum(case when current_proc_dt - lead_dt <=6 then issued_policy else 0 end ) Day7,
state_cd,
case when state_cd in('CT','ME','NY','PA','RI','VT','NH','MA') then 'Target'
     when state_cd in('AZ','DC','DE','MD','NC','NM','OR','VA','WA') then 'Tier2'
else 'Non-Target' 
end state_grp,
to_char(lead_dt,'YYYY-MM') lead_monyr
 
from  PROD_DB_PII_ANALYTICS.SANDBOX.leads_policies_auto_home_pup  a,
PROD_DB_PII_ANALYTICS.SANDBOX.last_touch_attribution_vq b
where a.lead_Dt between'{start_date}' AND '{end_date}'
and b.medium in('cpc','Paid Search')
and a.lead_storage_nbr=b.orig_storage_no
group by 1,2,3,4,5,6,7,8,9,10,16,17,18;

'''

internal_data = pd.read_sql(query2, ctx)
print(internal_data.shape)
internal_data.head()

/tmp/ipykernel_3198170/1854016874.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  internal_data = pd.read_sql(query2, ctx)


(436217, 18)


,LEAD_STORAGE_NBR,LINE_OF_BUSINESS,QUALITY_SEG,LEAD_DT,LEAD_CONTACT_METHOD,LEAD_MEDIUM,MEDIUM_CHANNEL,MEDIUM,SOURCE,CAMPAIGN,LEAD,QUOTE,ISSUE,DAY0,DAY7,STATE_CD,STATE_GRP,LEAD_MONYR
0,W011391783,AUTOMOBILE,5,2025-02-21,WEB,Paid Search,Paid Search,cpc,google,auto generic ct performance max,1,1,0,0,0,CT,Target,2025-02
1,W012273074,AUTOMOBILE,1,2025-09-30,WEB,Paid Search,Paid Search,cpc,google,auto generic - ny,1,1,0,0,0,NY,Target,2025-09
2,W011950241,AUTOMOBILE,1,2025-07-14,WEB,Paid Search,Paid Search,cpc,google,home generic - ct,1,1,0,0,0,NY,Target,2025-07
3,W013279044,AUTOMOBILE,5,2026-06-14,WEB,Paid Search,Paid Search,cpc,google,auto generic - az,1,1,0,0,0,AZ,Tier2,2026-06
4,W012467296,AUTOMOBILE,1,2025-11-18,WEB,Paid Search,Paid Search,cpc,google,enterprise - strategic,1,1,1,0,0,WA,Tier2,2025-11


In [50]:
#internal_data = pd.read_csv('/data/Projects/lead_policy_counts/aggregated_df_search.csv')

In [51]:
internal_data.columns = internal_data.columns.str.lower()

In [52]:
internal_data = internal_data[internal_data['lead_contact_method'] == 'PHONE']

In [53]:
exclude_list = [
    'QS_AUTO_INSURANCE', 'MA_AUTO_INSURANCE', 'MA_HOME_INSURANCE', 'CA_AUTO_INSURANCE',
    'USNEWS_HOME_INSURANCE', 'CNBC', 'CA_HOME_INSURANCE', 'MONEYGROUP_TIER1PUBS',
    'BEST_MONEY_AUTO_INSURANCE', 'MONEY.COM', '2501R'
]

internal_data = internal_data[
    ~internal_data['campaign'].str.upper().isin(exclude_list)
]

In [54]:

import pandas as pd

def update_campaign_name(df: pd.DataFrame) -> pd.DataFrame:
    """
    Update df['campaign'] based on specific rules involving 'campaign' and 'state_cd'.
    Rules are applied in sequence. If a rule does not match, the campaign stays as-is.

    Required columns: 'campaign', 'state_cd'
    """

    # --- Validate required columns ---
    required = {'campaign', 'state_cd'}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    # Work on a copy (order of assignments controls precedence)
    out = df['campaign'].copy()

    # --- Shared state sets ---
    states_ct_az = {'CT', 'AZ'}
    strategic_states = {'AZ', 'CT', 'DC', 'DE', 'MA', 'MD', 'ME', 'NH', 'NM', 'NY', 'OR', 'PA', 'RI', 'VT', 'WA', 'VA','NC'}

    # --- Campaign groups from earlier rules ---
    auto_pdps = {
        'Amica.com Auto EAS - Google',
        'Amica.com Auto EAS - Bing',  # exact per your spec
    }
    home_pdps = {
        'Amica.com Home EAS - Google',
        'Amica.com Home EAS - Bing',
    }
    enterprise_campaigns = {
        'Enterprise Generic - Google',
        'Brand - Google - Desktop',
        'Enterprise Paid Search Site-desktop only',
        'Enterprise Call Extension - Google',
        'Paid Search Bundle Message Test - Convenience message',
    }
    auto_conv = {
        'Auto Generic - Google',
        'Auto Call Extension - Google',
        'Auto Generic - Bing',
        'Auto Call Extension - Bing',
    }

    # --- New campaign names (previous additions) ---
    auto_bundle_google = 'Bundle Campaign - Google'
    auto_bundle_bing   = 'Bundle Campaign - Bing'
    pmax_home_campaign = 'PMAX Home Campaign - Google'
    bing_enterprise_campaigns = {
        'Enterprise Call Extension - Bing',
        'Enterprise Generic - Bing',
    }

    # ============================================================
    # 1) PDPs (Auto): CT/AZ -> AUTO GENERIC - {state}
    # ============================================================
    m1 = df['campaign'].isin(auto_pdps) & df['state_cd'].isin(states_ct_az)
    out.loc[m1] = 'AUTO GENERIC - ' + df.loc[m1, 'state_cd']

    # ============================================================
    # 2) PDPs (Home): CT/AZ -> HOME GENERIC - {state}
    # ============================================================
    m2 = df['campaign'].isin(home_pdps) & df['state_cd'].isin(states_ct_az)
    out.loc[m2] = 'HOME GENERIC - ' + df.loc[m2, 'state_cd']

    # ============================================================
    # 3) Enterprise (non-Bing)
    # ============================================================
    m3 = df['campaign'].isin(enterprise_campaigns)
    m3_strat = m3 & df['state_cd'].isin(strategic_states)
    m3_non   = m3 & ~df['state_cd'].isin(strategic_states)
    out.loc[m3_strat] = 'AMICA BRAND - ENTERPRISE - STRATEGIC'
    out.loc[m3_non]   = 'AMICA BRAND - ENTERPRISE - NON-STRATEGIC'

    # ============================================================
    # 4) Auto conversion tracking -> AUTO GENERIC - {state}
    # ============================================================
    m4 = df['campaign'].isin(auto_conv) & df['state_cd'].isin(strategic_states)
    out.loc[m4] = 'AUTO GENERIC - ' + df.loc[m4, 'state_cd']

    # ============================================================
    # 5) Home conversion tracking (split into non-Bing and Bing)
    #    Non-Bing -> HOME GENERIC - {state}
    #    Bing     -> HOME GENERIC - {state} - BING
    # ============================================================
    home_conv_non_bing = {
        'Home Generic - Google',
        'Home Generic',
        'Home Call Extension - Google',
    }
    home_conv_bing = {
        'Home Generic - Bing',
        'Home Call Extension - Bing',
    }
    m5a = df['campaign'].isin(home_conv_non_bing) & df['state_cd'].isin(strategic_states)
    out.loc[m5a] = 'HOME GENERIC - ' + df.loc[m5a, 'state_cd']
    m5b = df['campaign'].isin(home_conv_bing) & df['state_cd'].isin(strategic_states)
    out.loc[m5b] = 'HOME GENERIC - ' + df.loc[m5b, 'state_cd'] + ' - BING'

    # ============================================================
    # 6) Auto Bundle (Google): PA/NH/RI/CT -> Auto Generic - {state} - Bundle
    # ============================================================
    m6 = (df['campaign'] == auto_bundle_google) & df['state_cd'].isin({'PA', 'NH', 'RI', 'CT'})
    out.loc[m6] = 'Auto Generic - ' + df.loc[m6, 'state_cd'] + ' - Bundle'

    # ============================================================
    # 7) Auto Bundle (Bing):
    #    PA/CT -> Auto Generic - {state} - Bundle
    #    NH/RI -> Auto Generic - NH | RI - Bundle
    # ============================================================
    m7a = (df['campaign'] == auto_bundle_bing) & df['state_cd'].isin({'PA', 'CT'})
    out.loc[m7a] = 'Auto Generic - ' + df.loc[m7a, 'state_cd'] + ' - Bundle'
    m7b = (df['campaign'] == auto_bundle_bing) & df['state_cd'].isin({'NH', 'RI'})
    out.loc[m7b] = 'Auto Generic - NH | RI - Bundle'

    # ============================================================
    # 8) PMAX Home campaign:
    #    NH/MA/VT -> Home Generic - {state}- Performance Max (no space before hyphen)
    # ============================================================
    m8 = (df['campaign'] == pmax_home_campaign) & df['state_cd'].isin({'NH', 'MA', 'VT','CT','RI'})
    out.loc[m8] = 'Home Generic - ' + df.loc[m8, 'state_cd'] + ' - Performance Max'

    # ============================================================
    # 9) Bing Enterprise group:
    #    strategic -> Amica Brand - Enteprise - Strategic - Bing
    #    else      -> Amica Brand - Enteprise - Non Strategic - Bing
    # ============================================================
    m9 = df['campaign'].isin(bing_enterprise_campaigns)
    m9_strat = m9 & df['state_cd'].isin(strategic_states)
    m9_non   = m9 & ~df['state_cd'].isin(strategic_states)
    out.loc[m9_strat] = 'AMICA BRAND - ENTERPRISE - STRATEGIC - BING'
    out.loc[m9_non]   = 'AMICA BRAND - ENTERPRISE - NON STRATEGIC - BING'

    # ============================================================
    # 10) NEW: Brand renames (Google/Bing)
    # ============================================================
    m10a = (df['campaign'] == 'Amica Auto Brand campaign - Google')
    out.loc[m10a] = 'Amica Auto - Brand'
    m10b = (df['campaign'] == 'Amica Home Brand campaign - Google')
    out.loc[m10b] = 'Amica Brand - Home Insurance'
    m10c = (df['campaign'] == 'Amica Auto Brand - Bing')
    out.loc[m10c] = 'Amica Auto - Brand - Bing'
    m10d = (df['campaign'] == 'Amica Home Brand - Bing')
    out.loc[m10d] = 'Amica Brand - Home Insurance - Bing'

    # ============================================================
    # 11) NEW: PMAX states logic for two campaign names
    # ============================================================
    pmax_states = {'CT', 'MA', 'WA', 'RI', 'NY', 'NH', 'ME', 'VT', 'PA'}
    pmax_campaigns = {'AUTO GENERIC - PERFORMANCE MAX', 'Pmax Auto Campaign - Google'}
    m11 = df['campaign'].isin(pmax_campaigns) & df['state_cd'].isin(pmax_states)
    out.loc[m11] = 'AUTO GENERIC - ' + df.loc[m11, 'state_cd'] + ' - PERFORMANCE MAX'

    # ============================================================
    # 12) NEW: Condo Call Extension - Conversion Tracking -> Condo Insurance - Generic
    # ============================================================
    m12 = (df['campaign'] == 'Condo Call Extension - Google')
    out.loc[m12] = 'Condo Insurance - Generic'

    # ============================================================
    # 13) NEW: LHHI state-specific renames
    # ============================================================
    campaign_norm = df['campaign'].fillna('').str.strip().str.upper()
    state_norm = df['state_cd'].fillna('').str.strip().str.upper()

    m13_auto = (
    campaign_norm.eq('AUTO GENERIC - LHHI - GOOGLE')
    & state_norm.isin({'DC', 'MA', 'PA', 'CT', 'RI'}))
    out.loc[m13_auto] = 'AUTO GENERIC - ' + state_norm[m13_auto] + ' - LHHI'

    m13_home = (
    campaign_norm.eq('HOME GENERIC - LHHI - GOOGLE')
    & state_norm.isin({'DC', 'MA', 'CT', 'RI'}))
    out.loc[m13_home] = 'HOME GENERIC - ' + state_norm[m13_home] + ' - LHHI'

     # ============================================================
    # 14) NEW: Condo Generic renames -> CONDO INSURANCE - GENERIC
    # ============================================================
    condo_generic_campaigns = {
        'CONDO GENERIC',
        'CONDO GENERIC - GOOGLE',
        'CONDO - GENERIC - STRATEGIC',
    }
    m14 = campaign_norm.isin(condo_generic_campaigns)
    out.loc[m14] = 'CONDO INSURANCE - GENERIC'

    # --- Write back and return ---
    df = df.copy()
    df['campaign'] = out
    return df


In [55]:
internal_data = update_campaign_name(internal_data)

In [56]:
start_date1 = pd.to_datetime(start_date1)
end_date1 = pd.to_datetime(end_date1)
internal_data['lead_dt'] = pd.to_datetime(internal_data['lead_dt'])
internal_data = internal_data[(internal_data['lead_dt'] >= start_date1) &(internal_data['lead_dt'] <= end_date1)]
internal_data = internal_data[internal_data['medium'].isin(['Paid Search', 'cpc'])]
internal_data.rename(columns={'state_cd': 'state'}, inplace=True)


In [57]:
internal_data['lead_dt'] = pd.to_datetime(internal_data['lead_dt'])
internal_data['lead_monyr'] = internal_data['lead_dt'].dt.strftime('%Y-%m')

In [58]:
internal_data['campaign'] = internal_data['campaign'].astype(str).str.upper()

In [59]:
def classify_keyword(campaign):
    campaign_upper = str(campaign).upper()
    if 'BRAND' in campaign_upper or 'ENTERPRISE' in campaign_upper:
        return 'Brand'
    elif 'MAX' in campaign_upper:
        return 'PMAX'
    else:
        return 'Generic'

internal_data['keyword_type'] = internal_data['campaign'].apply(classify_keyword)

In [60]:
internal_data['Auto_Phone_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE' else None,axis=1)
internal_data['Auto_Web_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB' else None,axis=1)
internal_data['Seg_1_2_Auto_Web_leads'] = internal_data.apply(lambda row: row['Auto_Web_leads'] if row['quality_seg'] in [1, 2] else 0,axis=1)
internal_data['Seg_0_Auto_Web_leads'] = internal_data.apply(lambda row: 1 if (row['line_of_business'] == 'AUTOMOBILE' and row['quality_seg'] == 0 and row['lead_contact_method'] == 'WEB') else 0,axis=1)
internal_data['Seg_1_2_Auto_Phone_leads'] = internal_data.apply(lambda row: row['Auto_Phone_leads'] if row['quality_seg'] in [1, 2] else 0,axis=1)
internal_data['Seg_0_Auto_Phone_leads'] = internal_data.apply(lambda row: 1 if (row['line_of_business'] == 'AUTOMOBILE' and row['quality_seg'] == 0 and row['lead_contact_method'] == 'PHONE') else 0,axis=1)
internal_data['Auto_Issues'] = internal_data.apply(lambda row: row['issue'] if row['line_of_business'] == 'AUTOMOBILE' else None,axis=1)
internal_data['Auto_Web_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB') else None, axis=1)
internal_data['Auto_Phone_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE') else None, axis=1)
internal_data['Auto_Day7'] = internal_data.apply(lambda row: row['day7'] if row['line_of_business'] == 'AUTOMOBILE' else None,axis=1)
internal_data['Auto_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Auto_Phone_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'AUTOMOBILE' and row['lead_contact_method'] == 'PHONE') else None,axis=1)

In [61]:
internal_data['Home_Phone_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE' else None,axis=1)
internal_data['Home_Web_leads'] = internal_data.apply(lambda row: row['lead'] if row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB' else None,axis=1)
internal_data['Home_Issues'] = internal_data.apply(lambda row: row['issue'] if row['line_of_business'] == 'HOMEOWNERS' else None,axis=1)
internal_data['Home_Web_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Home_Phone_Issues'] = internal_data.apply(lambda row: row['issue'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE') else None,axis=1)
internal_data['Home_Day7'] = internal_data.apply(lambda row: row['day7'] if row['line_of_business'] == 'HOMEOWNERS' else None,axis=1)
internal_data['Home_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'WEB') else None,axis=1)
internal_data['Home_Web_Day7'] = internal_data.apply(lambda row: row['day7'] if (row['line_of_business'] == 'HOMEOWNERS' and row['lead_contact_method'] == 'PHONE') else None,axis=1)

In [62]:
# Assume `df` is your main dataset and `lookup_df` is your lookup table

# Create a dictionary from the lookup table
lookup_dict = dict(zip(lookup_df['int_names'], lookup_df['eng_names']))

# Replace campaign names using the dictionary
internal_data['campaign'] = internal_data['campaign'].apply(lambda x: lookup_dict.get(x, x))

In [63]:


internal_data['Engine'] = internal_data.apply(
    lambda row: (
        'bing'
        # safe check: convert campaign to string, then lowercase
        if 'bing' in str(row['campaign']).lower()
        else row['source'] if row['source'] in ['google', 'bing']
        else 'google' if row['source'] == 'google.com'
        else 'bing' if row['source'] == 'bing.com'
        else 'google'
    ),
    axis=1
)


In [64]:
# Group Google Ads data
google_grouped = google_data.groupby(['lead_monyr', 'state', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
    'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

# Group Google Ads data
bing_grouped = bing_data.groupby(['lead_monyr', 'state', 'CAMPAIGN_NAME','keyword_type','Engine'], as_index=False).agg({
    'IMPRESSIONS': 'sum',
    'SPEND': 'sum',
    'CLICKS': 'sum',
    'CONVERSIONS': 'sum'
})

# Group Internal Company data
internal_grouped = internal_data.groupby(['lead_monyr', 'state', 'campaign','keyword_type','Engine'], as_index=False).agg({
     'lead': 'sum',
    'quote': 'sum',
    'issue': 'sum',
    'day0': 'sum',
    'day7': 'sum',
    'Auto_Phone_leads' :'sum',
    'Auto_Web_leads':'sum',
    'Seg_1_2_Auto_Web_leads':'sum',
    'Seg_0_Auto_Web_leads':'sum',
    'Seg_1_2_Auto_Phone_leads':'sum',
    'Seg_0_Auto_Phone_leads':'sum',
    'Auto_Issues':'sum',
    'Home_Phone_leads':'sum',
    'Home_Web_leads':'sum',
    'Home_Issues':'sum',
    'Auto_Day7':'sum',
    'Home_Day7':'sum',
    'Auto_Web_Issues' :'sum',
    'Auto_Phone_Issues' :'sum',
    'Auto_Web_Day7' :'sum',
    'Auto_Web_Day7' :'sum',
    'Home_Web_Issues' :'sum',
    'Home_Phone_Issues' :'sum',
    'Home_Web_Day7' :'sum',
    'Home_Web_Day7' :'sum'
})


In [65]:
# Convert campaign names to uppercase
engine['CAMPAIGN_NAME'] = engine['CAMPAIGN_NAME'].str.upper()
engine['Engine'] = engine['Engine'].str.upper()
internal_grouped['campaign'] = internal_grouped['campaign'].str.upper()
internal_grouped['Engine'] = internal_grouped['Engine'].str.upper()

In [66]:
exact_merged = pd.merge(engine, internal_grouped, left_on=['lead_monyr', 'state', 'CAMPAIGN_NAME', 'keyword_type', 'Engine'],right_on=['lead_monyr', 'state', 'campaign', 'keyword_type', 'Engine'], how='inner')


In [67]:

exact_merged_test = pd.merge(
    engine,
    internal_grouped,
    left_on=['lead_monyr', 'state', 'CAMPAIGN_NAME', 'keyword_type', 'Engine'],
    right_on=['lead_monyr', 'state', 'campaign', 'keyword_type', 'Engine'],
    how='outer',  # use 'outer' to capture all unmatched rows
    indicator=True
)


In [68]:
engine_unmatched = exact_merged_test[exact_merged_test['_merge'] == 'left_only']
internal_unmatched = exact_merged_test[exact_merged_test['_merge'] == 'right_only']

In [69]:
exact_merged = exact_merged.drop(columns=['campaign'])
engine_unmatched = engine_unmatched.drop(columns=['_merge'])
internal_unmatched = internal_unmatched.drop(columns=['_merge'])
internal_unmatched = internal_unmatched.drop(columns=['CAMPAIGN_NAME'])

In [70]:
internal_unmatched.rename(columns={
    'campaign': 'CAMPAIGN_NAME'
}, inplace=True)

In [71]:
internal_unmatched = internal_unmatched[['lead_monyr','state','CAMPAIGN_NAME', 'keyword_type', 'Engine', 'IMPRESSIONS', 'SPEND',
       'CLICKS', 'CONVERSIONS', 'lead', 'quote', 'issue',
       'day0', 'day7', 'Auto_Phone_leads', 'Auto_Web_leads',
       'Seg_1_2_Auto_Web_leads', 'Seg_0_Auto_Web_leads',
       'Seg_1_2_Auto_Phone_leads', 'Seg_0_Auto_Phone_leads', 'Auto_Issues',
       'Home_Phone_leads', 'Home_Web_leads', 'Home_Issues', 'Auto_Day7',
       'Home_Day7',
 'Auto_Web_Issues', 'Auto_Phone_Issues', 'Auto_Web_Day7',
    'Home_Web_Issues', 'Home_Phone_Issues', 'Home_Web_Day7'
]]


In [72]:
exact_merged[['IMPRESSIONS', 'SPEND','CLICKS','CONVERSIONS']] = 0

In [73]:
Search_Phone = pd.concat([exact_merged, internal_unmatched], ignore_index=True)

In [74]:
Search_Phone = Search_Phone.fillna(0)

/tmp/ipykernel_3198170/2545590212.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Search_Phone = Search_Phone.fillna(0)


In [75]:
Search_report = pd.concat([Search_Web, Search_Phone], ignore_index=True)

In [76]:
Search_report = Search_report.fillna(0)

In [77]:
# Add new column based on logic
def classify(value):
    value = str(value).upper()
    if "AUTO" in value:
        return "Auto"
    elif "HOME" in value or "CONDO" in value:
        return "Home"
    elif "ENTERPRISE" in value:
        return "Enterprise"
    else:
        return ""
Search_report['Category'] = Search_report['CAMPAIGN_NAME'].apply(classify)


In [78]:
Search_report.rename(columns={
    'lead_monyr': 'Year_Month',
    'CAMPAIGN_NAME': 'Campaign',
    'keyword_type': 'Keyword_type',
    'IMPRESSIONS': 'Impressions',
    'SPEND': 'Spend',
    'CLICKS': 'Clicks',
    'CONVERSIONS' : 'Conversions',
    'lead': 'Lead',
    'quote': 'Quote',
    'issue': 'Issue',
    'day0': 'Day0',
    'day7': 'Day7'
    
}, inplace=True)

In [79]:
Search_report['dev_issues'] = Search_report.apply(lambda row: row['Day7'] / 0.55 if row['Year_Month'] >= '2026-4' else row['Issue'],axis=1)

In [80]:

def classify_campaign(campaign):
    campaign = str(campaign).upper()
    if "AUTO" in campaign:
        return "Auto"
    elif "HOME" in campaign or "CONDO" in campaign:
        return "Home"
    elif "ENTERPRISE" in campaign:
        return "Enterprise"
    else:
        return ""

Search_report['Account'] = Search_report['Campaign'].apply(classify_campaign)

In [81]:
#Search_report.to_csv('/data/Users/Shelly/Search Report/Search_Mutual_Report_campaign_level.csv')

### attribution logic

In [82]:
phone_re = pd.read_excel('/data/Users/Shelly/Search_Report/phone_reattribution.xlsx')

In [83]:

phone_lookup = pd.merge(
    exact_merged,
    phone_re,
    left_on=['CAMPAIGN_NAME'],
    right_on=['Campaign2'],
    how='left'
)


In [84]:

import pandas as pd

# columns you want to normalize within each group and overwrite
cols = ['lead', 'quote', 'issue', 'day0',
       'day7', 'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
       'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
       'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
       'Home_Web_leads', 'Home_Issues', 'Auto_Day7', 'Home_Day7',
       'Auto_Web_Issues', 'Auto_Phone_Issues', 'Auto_Web_Day7',
       'Home_Web_Issues', 'Home_Phone_Issues', 'Home_Web_Day7']
group_cols = ['lead_monyr', 'state', 'Campaign1']

# compute per-group sums for those columns
sums = (
    phone_lookup
    .groupby(group_cols)[cols]
    .transform('sum')
    .replace(0, pd.NA)  # avoid division by zero (optional)
)

# overwrite the original columns with their normalized values
phone_lookup[cols] = phone_lookup[cols].div(sums)

# rename the overwritten columns to add the "_factor" suffix
phone_lookup.rename(columns={c: f"{c}_factor" for c in cols}, inplace=True)

# (optional) if you prefer zeros instead of NA for groups that summed to 0:
phone_lookup[[f"{c}_factor" for c in cols]] = phone_lookup[[f"{c}_factor" for c in cols]].fillna(0)


/tmp/ipykernel_3198170/1005002616.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  phone_lookup[[f"{c}_factor" for c in cols]] = phone_lookup[[f"{c}_factor" for c in cols]].fillna(0)


In [85]:

phone_lookup = phone_lookup.drop(columns=['Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS'])


In [86]:

internal_unmatched = internal_unmatched.drop(columns=['Engine', 'IMPRESSIONS',
       'SPEND', 'CLICKS', 'CONVERSIONS'])

In [87]:

reattributed = pd.merge(
    phone_lookup,
    internal_unmatched,
    left_on=['lead_monyr','state','Campaign1'],
    right_on=['lead_monyr','state','CAMPAIGN_NAME'],
    how='left'
)


In [88]:

import pandas as pd

# ---- 1) Define the columns you mentioned ----
factor_cols = [
    'lead_factor', 'quote_factor', 'issue_factor', 'day0_factor', 'day7_factor',
    'Auto_Phone_leads_factor', 'Auto_Web_leads_factor',
    'Seg_1_2_Auto_Web_leads_factor', 'Seg_0_Auto_Web_leads_factor',
    'Seg_1_2_Auto_Phone_leads_factor', 'Seg_0_Auto_Phone_leads_factor',
    'Auto_Issues_factor', 'Home_Phone_leads_factor',
    'Home_Web_leads_factor', 'Home_Issues_factor', 'Auto_Day7_factor',
    'Home_Day7_factor', 'Auto_Web_Issues_factor',
    'Auto_Phone_Issues_factor', 'Auto_Web_Day7_factor',
    'Home_Web_Issues_factor', 'Home_Phone_Issues_factor',
    'Home_Web_Day7_factor'
]

base_cols = [
    'lead', 'quote', 'issue', 'day0', 'day7',
    'Auto_Phone_leads', 'Auto_Web_leads', 'Seg_1_2_Auto_Web_leads',
    'Seg_0_Auto_Web_leads', 'Seg_1_2_Auto_Phone_leads',
    'Seg_0_Auto_Phone_leads', 'Auto_Issues', 'Home_Phone_leads',
    'Home_Web_leads', 'Home_Issues', 'Auto_Day7', 'Home_Day7','Auto_Web_Issues','Auto_Phone_Issues','Auto_Web_Day7',
    'Home_Web_Issues',	'Home_Phone_Issues','Home_Web_Day7'

]

# ---- 2) Build (base, factor) pairs where both columns actually exist ----
pairs = []
for f in factor_cols:
    if f.endswith('_factor'):
        base = f[:-7]  # strip the "_factor" suffix
        if base in reattributed.columns and f in reattributed.columns:
            pairs.append((base, f))

# If you only want to compute for the explicitly provided base_cols, filter to those:
# pairs = [(b, f) for (b, f) in pairs if b in set(base_cols)]

# ---- 3) Create the new columns (UPPERCASE names) as base * factor ----
# Notes:
# - If you want NaNs where either side is missing, this is fine as-is.
# - If you want to treat missing factor as 0, you could fillna on the factor part first.
for base, factor in pairs:
    new_col = base.upper()  # e.g. 'lead' -> 'LEAD'
    reattributed[new_col] = reattributed[base] * reattributed[factor].round(2)

# ---- 4) Drop all original base and factor columns you listed ----
# Safely intersect with existing columns so drop() won't error.
to_drop = list(set(factor_cols + base_cols) & set(reattributed.columns))
reattributed.drop(columns=to_drop, inplace=True)

# (Optional) If you want missing products to be 0 instead of NaN:
product_cols = [b.upper() for (b, f) in pairs]
reattributed[product_cols] = reattributed[product_cols].fillna(0).round(2)


/tmp/ipykernel_3198170/3570333837.py:53: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  reattributed[product_cols] = reattributed[product_cols].fillna(0).round(2)


In [89]:
#internal_unmatched_filtered.to_csv('/data/Users/Shelly/Search Report/internal_unmatched_filtered.csv')

In [90]:

to_remove = {
    'HOME GENERIC - GOOGLE',
    'HOME GENERIC',
    'AUTO GENERIC - GOOGLE',
    'HOME CALL EXTENSION - GOOGLE',
    'AUTO GENERIC - BING',
    'AMICA.COM HOME EAS - GOOGLE',
    'AUTO CALL EXTENSION - GOOGLE',
    'PMAX AUTO CAMPAIGN - GOOGLE',
    'AUTO GENERIC - CT - BUNDLE'  
}

# Keep only rows where campaign is NOT in the list
internal_unmatched_filtered = internal_unmatched[~internal_unmatched['CAMPAIGN_NAME'].isin(to_remove)]

reattributed_filtered = reattributed[reattributed['Campaign1'].isin(to_remove)]

In [91]:
Search_Phone = pd.concat([exact_merged, internal_unmatched_filtered], ignore_index=True)

In [92]:
Search_report = pd.concat([Search_Web, Search_Phone], ignore_index=True)

In [93]:
Search_report = Search_report.fillna(0)

/tmp/ipykernel_3198170/452573369.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Search_report = Search_report.fillna(0)


In [94]:
reattributed_filtered.to_csv('/data/Users/Shelly/Search Report/reattributed_filtered.csv')

In [95]:
reattributed_filtered = reattributed_filtered.drop(columns=["Campaign1", "Campaign2", "CAMPAIGN_NAME_y","keyword_type_y"])

In [96]:
reattributed_filtered["Engine"] = np.where(
    reattributed_filtered["CAMPAIGN_NAME_x"].str.contains("bing", case=False, na=False),
    "BING",
    "GOOGLE"
)

In [97]:
reattributed_filtered["Impressions"] = 0
reattributed_filtered["Spend"] = 0
reattributed_filtered["Clicks"] = 0
reattributed_filtered["Conversions"] = 0

In [98]:
reattributed_filtered["dev_issues"] = np.where(reattributed_filtered["lead_monyr"] > "2025-11", reattributed_filtered["DAY7"] / 0.55, reattributed_filtered["DAY0"])

In [99]:

c = reattributed_filtered["CAMPAIGN_NAME_x"].astype(str)

conditions = [
    c.str.contains("AUTO", case=False, na=False),
    c.str.contains("HOME|CONDO", case=False, na=False, regex=True),
    c.str.contains("ENTERPRISE", case=False, na=False),
]
choices = ["Auto", "Home", "Enterprise"]

reattributed_filtered["Account"] = np.select(conditions, choices, default="")

In [100]:
reattributed_filtered["Category"] = np.select(conditions, choices, default="")

In [101]:
# old_name : new_name
rename_map = {
    "lead_monyr": "Year_Month",
    "CAMPAIGN_NAME_x": "Campaign",
    "keyword_type_x": "Keyword_type",
    "LEAD": "Lead",
     "QUOTE": "Quote",
    "ISSUE": "Issue",
    "DAY0": "Day0",
     "DAY7": "Day7",
    "AUTO_PHONE_LEADS": "Auto_Phone_leads",
    "AUTO_WEB_LEADS": "Auto_Web_leads",
     "SEG_1_2_AUTO_WEB_LEADS": "Seg_1_2_Auto_Web_leads",
    "SEG_0_AUTO_WEB_LEADS": "Seg_0_Auto_Web_leads",
     "SEG_1_2_AUTO_PHONE_LEADS": "Seg_1_2_Auto_Phone_leads",
    "SEG_0_AUTO_PHONE_LEADS": "Seg_0_Auto_Phone_leads",
    "AUTO_WEB_LEADS": "Auto_Web_leads",
     "AUTO_ISSUES": "Auto_Issues",
    "HOME_PHONE_LEADS": "Home_Phone_leads",
     "HOME_WEB_LEADS": "Home_Web_leads",
    "HOME_ISSUES": "Home_Issues",
    "AUTO_DAY7": "Auto_Day7",
     "HOME_DAY7": "Home_Day7",
    "AUTO_WEB_ISSUES": "Auto_Web_Issues",
    "AUTO_PHONE_ISSUES": "Auto_Phone_Issues",
     "AUTO_WEB_DAY7": "Auto_Web_Day7",
    "HOME_WEB_ISSUES": "Home_Web_Issues",
    "HOME_PHONE_ISSUES": "Home_Phone_Issues",
     "HOME_WEB_DAY7": "Home_Web_Day7"
   
}

reattributed_filtered = reattributed_filtered.rename(columns=rename_map)


In [102]:
reattributed_filtered = reattributed_filtered[[	"Year_Month",
"state",
"Campaign",
"Keyword_type",
"Engine",
"Impressions",
"Spend",
"Clicks",
"Conversions",
"Lead",
"Quote",
"Issue",
"Day0",
"Day7",
"Auto_Phone_leads",
"Auto_Web_leads",
"Seg_1_2_Auto_Web_leads",
"Seg_0_Auto_Web_leads",
"Seg_1_2_Auto_Phone_leads",
"Seg_0_Auto_Phone_leads",
"Auto_Issues",
"Home_Phone_leads",
"Home_Web_leads",
"Home_Issues",
"Auto_Day7",
"Home_Day7",
"Auto_Web_Issues",
"Auto_Phone_Issues",
"Auto_Web_Day7",
"Home_Web_Issues",
"Home_Phone_Issues",
"Home_Web_Day7",
"Category",
"dev_issues",
"Account"
]]


In [103]:
Search_report = Search_report.fillna(0)
# Add new column based on logic
def classify(value):
    value = str(value).upper()
    if "AUTO" in value:
        return "Auto"
    elif "HOME" in value or "CONDO" in value:
        return "Home"
    elif "ENTERPRISE" in value:
        return "Enterprise"
    else:
        return ""
Search_report['Category'] = Search_report['CAMPAIGN_NAME'].apply(classify)

Search_report.rename(columns={
    'lead_monyr': 'Year_Month',
    'CAMPAIGN_NAME': 'Campaign',
    'keyword_type': 'Keyword_type',
    'IMPRESSIONS': 'Impressions',
    'SPEND': 'Spend',
    'CLICKS': 'Clicks',
    'CONVERSIONS' : 'Conversions',
    'lead': 'Lead',
    'quote': 'Quote',
    'issue': 'Issue',
    'day0': 'Day0',
    'day7': 'Day7'
    
}, inplace=True)

Search_report['dev_issues'] = Search_report.apply(lambda row: row['Day7'] / 0.55 if row['Year_Month'] >= '2026-4' else row['Issue'],axis=1)


def classify_campaign(campaign):
    campaign = str(campaign).upper()
    if "AUTO" in campaign:
        return "Auto"
    elif "HOME" in campaign or "CONDO" in campaign:
        return "Home"
    elif "ENTERPRISE" in campaign:
        return "Enterprise"
    else:
        return ""

Search_report['Account'] = Search_report['Campaign'].apply(classify_campaign)

In [104]:
# stack rows from two datasets
stacked = pd.concat([Search_report,reattributed_filtered], axis=0, ignore_index=True)

In [105]:
stacked.to_csv('/data/Users/Shelly/Search Report/Search_Mutual_Report_Campaign_level_stacked1.csv')